# 3A - Modeling su Dataset MACRO-ONLY

## 3.1 - Import e Caricamento dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Caricamento del dataset
df_macro_lagged = pd.read_csv('./data/df_macro_long_lagged.csv', index_col=0 )

print(f"✓ Dataset preprocessato caricato: {df_macro_lagged.shape[0]} righe e {df_macro_lagged.shape[1]} colonne")
print(f"✓ Colonne disponibili: {df_macro_lagged.columns.tolist()}")
print(f"NaN values per colonna:\n{df_macro_lagged.isna().sum()}")

✓ Dataset preprocessato caricato: 9505 righe e 66 colonne
✓ Colonne disponibili: ['ECB_Deposit_Rate', 'ECB_MRO_Rate', 'FED_Rate', 'VIX', 'ESI_Index', 'Euribor_3M', 'Euribor_1Y', 'HICP_Euroarea', 'interbank_stress_spread', 'short_yield_curve_slope', 'bce_fed_spread', 'ECB_Deposit_Rate_lag_7d', 'ECB_Deposit_Rate_lag_30d', 'ECB_Deposit_Rate_lag_60d', 'ECB_Deposit_Rate_lag_90d', 'ECB_Deposit_Rate_lag_120d', 'ECB_MRO_Rate_lag_7d', 'ECB_MRO_Rate_lag_30d', 'ECB_MRO_Rate_lag_60d', 'ECB_MRO_Rate_lag_90d', 'ECB_MRO_Rate_lag_120d', 'FED_Rate_lag_7d', 'FED_Rate_lag_30d', 'FED_Rate_lag_60d', 'FED_Rate_lag_90d', 'FED_Rate_lag_120d', 'VIX_lag_7d', 'VIX_lag_30d', 'VIX_lag_60d', 'VIX_lag_90d', 'VIX_lag_120d', 'ESI_Index_lag_7d', 'ESI_Index_lag_30d', 'ESI_Index_lag_60d', 'ESI_Index_lag_90d', 'ESI_Index_lag_120d', 'Euribor_3M_lag_7d', 'Euribor_3M_lag_30d', 'Euribor_3M_lag_60d', 'Euribor_3M_lag_90d', 'Euribor_3M_lag_120d', 'Euribor_1Y_lag_7d', 'Euribor_1Y_lag_30d', 'Euribor_1Y_lag_60d', 'Euribor_1Y_lag_90

## 3.2 - Target Definition

Decidiamo di considerate come forecast horizon 3 mesi. Un orizionte minore catturerebbe solo il rumore, i tassi di interesse non variano così rapidamente, proviamo a catturare i cambiamenti in un tempo che include almeno due riuonioni della bce (avvengonono ogni 6 settimane) e catturiamo almeno due rilasci del dati sull'inflazione (rilasciati con cadenza mensile)

In [31]:
FORECAST_HORIZON = 60

# Definiamo target variable
df_macro_lagged['Euribor_3M_target'] = df_macro_lagged['Euribor_3M'].shift(-FORECAST_HORIZON)

#Scartiamo le righe con valori NaN nella colonna target
df_macro_lagged = df_macro_lagged.dropna(subset=['Euribor_3M_target']).copy()

df_macro_lagged['target'] = (df_macro_lagged['Euribor_3M_target'] > df_macro_lagged['Euribor_3M']).astype(int)

print(f"\nTarget distribution:")
print(df_macro_lagged['target'].value_counts())
print(f"Classe imbalance ratio: {(df_macro_lagged['target'] == 1).sum() / (df_macro_lagged['target'] == 0).sum():.3f}")



Target distribution:
target
0    5154
1    4291
Name: count, dtype: int64
Classe imbalance ratio: 0.833


Siamo contenti con circa una perfetta distribuzione     

## 3.3 - Modello Naive: Persistenza statica

Valutiamo innanzi tutto la persistenza statica, ovvero quella in cui prevediamo che tra 90 giorni ci sia lo stesso tasso. Considerando che stiamo facendo classificazione direzionale (1=sale, 0=non sale), la nostra colonna target sarà composta da tutti 0.

In [32]:
# Isoliamo X and y
y_true = df_macro_lagged['target']

y_naive_static = np.zeros_like(y_true)  # Prevediamo sempre la classe 0 (Euribor_3M non aumenterà)
print("--- Baseline 1: Static Persistence ---")
print(f"\nClassification Report:\n{classification_report(y_true, y_naive_static, zero_division=0)}") # zero_division=0 per evitare warning in caso di classi non predette

--- Baseline 1: Static Persistence ---

Classification Report:
              precision    recall  f1-score   support

           0       0.55      1.00      0.71      5154
           1       0.00      0.00      0.00      4291

    accuracy                           0.55      9445
   macro avg       0.27      0.50      0.35      9445
weighted avg       0.30      0.55      0.39      9445



Questi dati sono in linea con la distribuzione della classe di maggioranza. I modelli Più avanzati dovranno battere il **55% di accuratezza**.

## 3.4 - Modello Naive: Momentum Persistance

Valutiamo ora la persistenza del trend: predice che Euribor_3M aumenterà se è già in aumento negli ultimi 60 giorni.

In [33]:
# Assumiamo di usare la stessa classe del lag a 60 giorni come previsione
# Scommetto 1 se il trend passato era in salita, 0 altrimenti

trend_lag_21 = df_macro_lagged['Euribor_3M'] > df_macro_lagged['Euribor_3M_lag_60d']
y_naive_trend = trend_lag_21.astype(int)

print("--- Baseline 2: Lagged Trend ---")
print(f"\nClassification Report:\n{classification_report(y_true, y_naive_trend, zero_division=0)}")
print(f"ROC-AUC: {roc_auc_score(y_true, y_naive_trend):.3f}")

--- Baseline 2: Lagged Trend ---

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.80      0.80      5154
           1       0.76      0.77      0.77      4291

    accuracy                           0.79      9445
   macro avg       0.78      0.78      0.78      9445
weighted avg       0.79      0.79      0.79      9445

ROC-AUC: 0.785


Abbiamo un'accuratezza del **78%** questo perchè la BCE segue il suo trend per anni quindi necessariamente è molto probabile che se sono saliti 90 giorni fà contineranno a farlo. Questo fenomeno si chiama **Forte Autocorrelazione** (Inerzia). Il modello "Trend" intercetta questa inerzia alla perfezione.

## 3.5 - Modello Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
import joblib

# Prepariamo i dati
columns_to_exclude = ['Euribor_3M_target', 'target']
feature_columns = [col for col in df_macro_lagged.columns if col not in columns_to_exclude]

X = df_macro_lagged[feature_columns].to_numpy(dtype=float)
y = df_macro_lagged['target'].to_numpy(dtype=int)

# TimeSeriesSplit con gap per evitare overlap leakage
tscv = TimeSeriesSplit(n_splits=5, gap=FORECAST_HORIZON)

# CREIAMO LA PIPELINE E LO SPAZIO DI RICERCA
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

param_dist = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [3, 5, 7, 10],
    'rf__min_samples_split': [2, 5, 10]
}

print("Avvio Ottimizzazione Iperparametri con RandomizedSearchCV...")

# RANDOMIZED SEARCH CV
random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=5,                   # Numero di combinazioni da provare (aumenta se hai tempo)
    scoring='roc_auc',
    cv=tscv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X, y)

print("\n" + "="*50)
print(f"Miglior ROC-AUC medio ottenuto: {random_search.best_score_:.3f}")
print("Migliori Iperparametri trovati:")
for param, value in random_search.best_params_.items():
    print(f" - {param.replace('rf__', '')}: {value}")

# Salviamo il modello vincitore per SHAP
joblib.dump(random_search.best_estimator_, './data/best_rf_macro_model.pkl')
print("✓ Modello Random Forest salvato con successo!")

Avvio Ottimizzazione Iperparametri con RandomizedSearchCV...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Miglior ROC-AUC medio ottenuto: 0.505
Migliori Iperparametri trovati:
 - n_estimators: 100
 - min_samples_split: 5
 - max_depth: 5
✓ Modello Random Forest salvato con successo!


## 3.6 - Modello LSTM (Long Short-Term Memory)

Importiamo il dataset e come per gli altri modelli creaiamo le colonne target

In [ ]:
# Importiamo il dataset macro senza lag
df_macro_pp = pd.read_csv('./data/df_macro_long.csv', index_col=0 )

FORECAST_HORIZON = 60

# Creiamo la colonna col valore futuro
df_macro_pp['Euribor_3M_target'] = df_macro_pp['Euribor_3M'].shift(-FORECAST_HORIZON)

# Droppiamo gli ultimi 90 giorni (che ora sono NaN) PRIMA di fare il test
df_macro_pp = df_macro_pp.dropna(subset=['Euribor_3M_target']).copy()

# Creiamo il target binario in totale sicurezza
df_macro_pp['target'] = (df_macro_pp['Euribor_3M_target'] > df_macro_pp['Euribor_3M']).astype(int)

# Isoliamo X e y
feature_cols = ['ECB_Deposit_Rate', 'ECB_MRO_Rate', 'FED_Rate', 'VIX', 'ESI_Index',
                'Euribor_3M', 'Euribor_1Y', 'HICP_Euroarea', 'interbank_stress_spread',
                'short_yield_curve_slope', 'bce_fed_spread']

X_base = df_macro_pp[feature_cols].values
y_base = df_macro_pp['target'].values

print(f"Shape di X: {X_base.shape}")
print(f"Shape di y: {y_base.shape}")


Shape di X: (9565, 11)
Shape di y: (9565,)


Ci occupiamo della funzione che crea la sliding window

In [36]:

# creiamo il tensore 3D per LSTM [samples, seq_length, features]
def create_sequences(X_data, y_data, seq_length):
    """
    Trasforma array 2D in array 3D [samples, seq_length, features].
    """
    xs, ys = [], []
    for i in range(len(X_data) - seq_length + 1): # Aggiunto +1 per non perdere l'ultima riga
        # Prende la finestra di giorni (es. da 0 a 59)
        xs.append(X_data[i : (i + seq_length)])

        # Il target DEVE essere quello associato all'ULTIMO giorno della finestra
        # L'ultimo giorno della finestra è (i + seq_length - 1)
        ys.append(y_data[i + seq_length - 1])

    return np.array(xs), np.array(ys)

SEQ_LENGTH = 180 # Quanti giorni nel passato LSTM deve guardare per fare la previsione


Creiamo la struttura della rete neurale

In [37]:
# Creiamo la classe LSTM
class MacroLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout_rate):
        super(MacroLSTM, self).__init__()

        # Se c'è 1 solo layer, diciamo a PyTorch che il dropout interno è 0.
        lstm_dropout = dropout_rate if num_layers > 1 else 0.0

        # layers LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=lstm_dropout) #  batch_first=True è FONDAMENTALE perché i nostri dati avranno forma (batch_size, seq_length, features)

        self.dropout = nn.Dropout(dropout_rate) # dropout per regolarizzazione

        # layer finale di classificazione
        self.fc = nn.Linear(hidden_size, 1)  # output binario
        self.sigmoid = nn.Sigmoid()  # per convertire l'output in probabilità

    def forward(self, x):
        # x shape: (batch, seq_len, features)
        lstm_out, (hn, cn) = self.lstm(x)  # lstm_out shape: (batch, seq_len, hidden_size)

        # Prendiamo solo l'output dell'ultimo timestep per la classificazione
        last_time_step_out = lstm_out[:, -1, :]  # shape: (batch, hidden_size)

        out = self.dropout(last_time_step_out)  # applichiamo dropout
        out = self.fc(out)  # shape: (batch, 1)
        return self.sigmoid(out)  # shape: (batch, 1) con valori tra 0 e 1

Prepariamo i dati per la k-fold assicurandoci di scalare i dati prima di creare le finestre 3d, ma dopo aver fatto lo split temporale altrimenti abbiamo data leakage

In [38]:
# Seleziono le colonne da usare come features
feature_base_cols = [col for col in df_macro_pp.columns if col != 'target' and col != 'Euribor_3M_target']

# Prepariamo la base
X_base = df_macro_pp[feature_base_cols].values
y_base = df_macro_pp['target'].values

tscv = TimeSeriesSplit(n_splits=5, gap=FORECAST_HORIZON) # Il gap è vitale!

n_splits = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

precomputed_folds = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_base)):
    print(f"\n--- Pre-processing FOLD {fold+1} ---")

    # Split 2D
    X_train_2d, X_test_2d = X_base[train_index], X_base[test_index]
    y_train_2d, y_test_2d = y_base[train_index], y_base[test_index]

    # Scaler (Fit SOLO sul train)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_2d)
    X_test_scaled = scaler.transform(X_test_2d)

    # Trasformazione in 3D (Creazione sequenze)
    X_train_3d, y_train_seq = create_sequences(X_train_scaled, y_train_2d, SEQ_LENGTH)
    X_test_3d, y_test_seq = create_sequences(X_test_scaled, y_test_2d, SEQ_LENGTH)

    # Conversione in Tensori PyTorch
    X_train_tensor = torch.tensor(X_train_3d, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32).unsqueeze(1) # Aggiungiamo dimensione per BCE Loss

    X_test_tensor = torch.tensor(X_test_3d, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32).unsqueeze(1) # Aggiungiamo dimensione per BCE Loss

    # DataLoaders (per addestrare a "pacchetti" e non saturare la RAM)
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=256, shuffle=False)

    input_dim = X_train_3d.shape[2]
    precomputed_folds.append({
        'train_loader': train_loader,
        'test_loader': test_loader,
        'input_dim': input_dim
    })

print("\n✓ Tutti i fold sono stati processati e salvati in memoria!")

Using device: cuda

--- Pre-processing FOLD 1 ---

--- Pre-processing FOLD 2 ---

--- Pre-processing FOLD 3 ---

--- Pre-processing FOLD 4 ---

--- Pre-processing FOLD 5 ---

✓ Tutti i fold sono stati processati e salvati in memoria!


### Funzione per Optuna
Creiamo la funzione che Optuna chiamerà ripetutamente. Questa funzione prenderà i dati salvati e cercherà l'architettura perfetta.

In [39]:
%pip install optuna

In [40]:
import optuna

def objective(trial):
    # 1. Chiediamo a Optuna di "suggerire" i parametri per questo tentativo
    hidden_size = trial.suggest_categorical('hidden_size', [16, 32, 64])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    epochs = trial.suggest_int('epochs', 10, 30)

    fold_roc_aucs = []

    # 2. Addestriamo il modello suggerito sui nostri 5 fold pre-calcolati
    for fold_data in precomputed_folds:
        model = MacroLSTM(
            input_size=fold_data['input_dim'],
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout_rate=dropout_rate
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.BCELoss()

        # Training
        for epoch in range(epochs):
            model.train()
            for batch_X, batch_y in fold_data['train_loader']:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(batch_X), batch_y)
                loss.backward()
                optimizer.step()

        # Evaluation
        model.eval()
        y_preds_prob = []
        y_trues = []
        with torch.no_grad():
            for batch_X, batch_y in fold_data['test_loader']:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                probs = model(batch_X)
                y_preds_prob.extend(probs.cpu().numpy().flatten())
                y_trues.extend(batch_y.cpu().numpy().flatten())

        # Calcoliamo il ROC-AUC del fold e lo salviamo
        roc = roc_auc_score(y_trues, y_preds_prob)
        fold_roc_aucs.append(roc)

    # 3. La funzione DEVE restituire il valore che vogliamo massimizzare
    return np.mean(fold_roc_aucs)

In [41]:
print("Avvio Ottimizzazione Iperparametri LSTM con Optuna...")

# Creiamo lo "Studio" dicendogli che vogliamo massimizzare il ROC-AUC
study = optuna.create_study(direction='maximize')

# Facciamo 15 tentativi (puoi metterci anche 50 se hai tempo su Colab)
study.optimize(objective, n_trials=15)

print("\n" + "="*50)
print(f"Miglior ROC-AUC medio ottenuto: {study.best_value:.3f}")
print("Migliori Iperparametri trovati:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-29 09:46:54,138] A new study created in memory with name: no-name-a15e18a1-26f7-43b5-bdea-1c17d9dc1524


Avvio Ottimizzazione Iperparametri LSTM con Optuna...


[I 2026-06-29 09:47:10,491] Trial 0 finished with value: 0.4480685315631524 and parameters: {'hidden_size': 16, 'num_layers': 3, 'dropout_rate': 0.18402343691499065, 'lr': 0.0053375056143247415, 'epochs': 12}. Best is trial 0 with value: 0.4480685315631524.
[I 2026-06-29 09:47:26,436] Trial 1 finished with value: 0.3835327431208257 and parameters: {'hidden_size': 16, 'num_layers': 2, 'dropout_rate': 0.49583208426305225, 'lr': 0.0006323559456511321, 'epochs': 21}. Best is trial 0 with value: 0.4480685315631524.
[I 2026-06-29 09:47:40,119] Trial 2 finished with value: 0.5429460206750882 and parameters: {'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.1364308461115262, 'lr': 0.00037794554370821867, 'epochs': 29}. Best is trial 2 with value: 0.5429460206750882.
[I 2026-06-29 09:47:49,909] Trial 3 finished with value: 0.37656133835992006 and parameters: {'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.3576797058123823, 'lr': 0.0007381306826971449, 'epochs': 21}. Best is trial 2 


Miglior ROC-AUC medio ottenuto: 0.653
Migliori Iperparametri trovati:
  hidden_size: 32
  num_layers: 1
  dropout_rate: 0.291292000891704
  lr: 1.0666999983019157e-05
  epochs: 24


### 3.6.1 - Addestramento del Modello LSTM Definitivo e Salvataggio
Utilizziamo i migliori iperparametri trovati da Optuna per calcolare le metriche
complete (incluso F1-Score con soglia rigorosa a 0.5 per evitare data leakage)
e salviamo il modello.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_curve

print("\n" + "="*50)
print("ADDESTRAMENTO MODELLO LSTM FINALE (BEST PARAMS)")
print("="*50)

# 1. Estraiamo i parametri vincenti da Optuna
best_hidden_size = study.best_params['hidden_size']
best_num_layers = study.best_params['num_layers']
best_dropout = study.best_params['dropout_rate']
best_lr = study.best_params['lr']
best_epochs = study.best_params['epochs']

final_roc_aucs = []
final_f1_scores = []

# 2. Ri-addestriamo sui 5 fold per estrarre le metriche complete
for fold, fold_data in enumerate(precomputed_folds):
    print(f"\n--- Training Final LSTM - Fold {fold+1} ---")

    # Inizializziamo il modello con i parametri ottimali
    model = MacroLSTM(
        input_size=fold_data['input_dim'],
        hidden_size=best_hidden_size,
        num_layers=best_num_layers,
        dropout_rate=best_dropout
    ).to(device)

    # Ricalcoliamo il peso per la Loss di questo specifico Fold
    y_train_flat = []
    for _, batch_y in fold_data['train_loader']:
        y_train_flat.extend(batch_y.numpy().flatten())
    y_train_flat = np.array(y_train_flat)

    weights = compute_class_weight('balanced', classes=np.unique(y_train_flat), y=y_train_flat)
    pos_weight = torch.tensor([weights[1] / weights[0]], dtype=torch.float32).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_lr, weight_decay=1e-4)

    # NOTA: MacroLSTM include già un layer Sigmoid in output.
    # nn.BCEWithLogitsLoss applica internamente una sua Sigmoid, il che significa
    # che viene applicata due volte. Considera di rimuovere "self.sigmoid(out)"
    # da MacroLSTM in futuro.
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Training
    for epoch in range(best_epochs):
        model.train()
        for batch_X, batch_y in fold_data['train_loader']:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # Evaluation sul TRAIN SET per la curva ROC
    model.eval()
    train_probs = []
    train_trues = []

    with torch.no_grad():
        # MODIFICA 1: Usiamo train_loader e popoliamo train_probs e train_trues
        for batch_X, batch_y in fold_data['train_loader']:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            probs = model(batch_X)
            train_probs.extend(probs.cpu().numpy().flatten())
            train_trues.extend(batch_y.cpu().numpy().flatten())

    # Calcolo della Curva ROC sul TRAIN SET
    fpr, tpr, thresholds = roc_curve(train_trues, train_probs)
    # Indice di Youden: massimizza la differenza tra True Positive Rate e False Positive Rate
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]

    # --- EVALUATION SUL TEST SET DEL FOLD ---
    y_preds_prob = []
    y_trues = []

    with torch.no_grad():
        for batch_X, batch_y in fold_data['test_loader']:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            # MODIFICA 2: Rimosso torch.sigmoid ridondante (è già in MacroLSTM)
            probs = model(batch_X)

            y_preds_prob.extend(probs.cpu().numpy().flatten())
            y_trues.extend(batch_y.cpu().numpy().flatten())

    # SOGLIA DINAMICA
    y_preds_class = [1 if p > optimal_threshold else 0 for p in y_preds_prob]

    # Calcolo metriche
    roc = roc_auc_score(y_trues, y_preds_prob)
    f1 = f1_score(y_trues, y_preds_class)

    final_roc_aucs.append(roc)
    final_f1_scores.append(f1)

    print(f"Fold {fold+1} -> ROC-AUC: {roc:.3f} | F1-Score: {f1:.3f} (Soglia Ottimale Youden: {optimal_threshold:.3f})")
    print(f"Confusion Matrix:\n{confusion_matrix(y_trues, y_preds_class)}")

    # Salviamo il modello dell'ultimo fold (ci servirà per eventuali test futuri)
    if fold == n_splits - 1:
        torch.save(model.state_dict(), './data/best_lstm_macro_model.pth')
        print("\n✓ Pesi del modello LSTM (ultimo fold) salvati con successo!")

# 3. Risultati Finali
print("\n" + "="*50)
print(f"RISULTATO FINALE LSTM OPTIMIZED (Media su {n_splits} folds):")
print(f"Mean ROC-AUC : {np.mean(final_roc_aucs):.3f}")
print(f"Mean F1-Score: {np.mean(final_f1_scores):.3f}")
print("="*50)


ADDESTRAMENTO MODELLO LSTM FINALE (BEST PARAMS)

--- Training Final LSTM - Fold 1 ---
Fold 1 -> ROC-AUC: 0.440 | F1-Score: 0.719 (Soglia Ottimale Youden: 0.488)
Confusion Matrix:
[[113 203]
 [368 731]]

--- Training Final LSTM - Fold 2 ---
Fold 2 -> ROC-AUC: 0.512 | F1-Score: 0.578 (Soglia Ottimale Youden: 0.468)
Confusion Matrix:
[[  0 840]
 [  0 575]]

--- Training Final LSTM - Fold 3 ---
Fold 3 -> ROC-AUC: 0.120 | F1-Score: 0.124 (Soglia Ottimale Youden: 0.458)
Confusion Matrix:
[[ 158 1000]
 [ 174   83]]

--- Training Final LSTM - Fold 4 ---
Fold 4 -> ROC-AUC: 0.728 | F1-Score: 0.631 (Soglia Ottimale Youden: 0.504)
Confusion Matrix:
[[367 470]
 [ 95 483]]

--- Training Final LSTM - Fold 5 ---
Fold 5 -> ROC-AUC: 0.821 | F1-Score: 0.649 (Soglia Ottimale Youden: 0.477)
Confusion Matrix:
[[  0 736]
 [  0 679]]

✓ Pesi del modello LSTM (ultimo fold) salvati con successo!

RISULTATO FINALE LSTM OPTIMIZED (Media su 5 folds):
Mean ROC-AUC : 0.524
Mean F1-Score: 0.540
